# LyboAI Mobile Edge Runtime — Agent Quickstart (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajandua20/lyboai-releases/blob/main/notebooks/lyboai_quickstart.ipynb)

Build a **private, on-device AI agent** from Python — against the *real* runtime that ships inside mobile apps.

In ~10 minutes you will:
1. create an agent with local knowledge (grounded Q&A with sources)
2. add a **workflow with human-in-the-loop approval**
3. **train** routing from labelled examples
4. build, **sign**, install, and **evaluate** a capability pack
5. export privacy-safe training data and verify the tamper-evident audit trail

Everything runs inside this Colab VM — no API keys, no cloud model calls. The runtime is offline-first by design: a deterministic model tier answers structured tasks even with no LLM attached.


## 1 · Setup (~30 s)
Downloads the prebuilt `lybo` runtime binary from the public GitHub release and installs the `lyboai` Python SDK from PyPI — no source build, no Rust toolchain.


In [ ]:
%%bash
set -e
LYBO_VERSION=v0.3.1   # https://github.com/rajandua20/lyboai-releases/releases
mkdir -p /content/bin
if [ ! -x /content/bin/lybo ]; then
  curl -fsSL -o /content/bin/lybo \
    https://github.com/rajandua20/lyboai-releases/releases/download/${LYBO_VERSION}/lybo-linux-x64
  chmod +x /content/bin/lybo
fi
/content/bin/lybo version
pip install -q lyboai
echo 'Setup complete ✔'


In [ ]:
import os
os.environ['LYBO_BIN'] = '/content/bin/lybo'
from lyboai import Client, PackBuilder, sign_pack, evaluate, train_triggers, export_routing_dataset
print('lyboai SDK ready')


## 2 · Create an agent with local knowledge
Knowledge is chunked and indexed **at install time**; retrieval is hybrid (keyword ⊕ embedding) and answers carry **source attribution** — the agent says *"I don't have that locally"* rather than inventing content.


In [ ]:
import tempfile
lybo = Client(data_dir=tempfile.mkdtemp(prefix='colab-agent-'), binary=os.environ['LYBO_BIN'])

lybo.install_knowledge(
    doc_id='manual', title='Field Manual',
    body='The seal on pump unit seven must be replaced every six months using part S-42. '
         'Torque the housing bolts to 24 Nm in a cross pattern. '
         'Hot work requires a same-day permit from the site supervisor.')

lybo.register_skill(
    skill_id='field.ask', name='Ask the manual', pattern='retrieval',
    triggers=['what does the manual say about'], keywords=['manual', 'lookup'])

session = lybo.start_session()
out = lybo.submit(session, 'what does the manual say about the pump seal')
print(out['output']['text'])
print('sources:', out['output']['sources'])


## 3 · A workflow with human-in-the-loop approval
Workflows are **declarative state graphs** (no code — safe to ship in signed packs). The `confirm` step **pauses the run** until a human decides; on device your app shows an approval sheet, here *you* are the human.


In [ ]:
lybo.register_workflow({
  'workflow_id': 'wf.report', 'version': '1', 'pack_id': 'app',
  'description': 'extract → approve → save', 'initial_state': 'extract',
  'max_steps': 24, 'auto_checkpoint': True,
  'states': {
    'extract': {'step': {'kind': 'extract', 'input_field': 'input',
                          'output_schema': {'type': 'object', 'required': ['asset_id'],
                                             'properties': {'asset_id': {'type': 'string'},
                                                             'condition': {'type': 'string'}}},
                          'output_field': 'report'},
                 'transitions': [{'when': {'kind': 'always'}, 'to': 'confirm'}]},
    'confirm': {'step': {'kind': 'confirm', 'message_template': 'Save report? {{report}}'},
                 'transitions': [{'when': {'kind': 'approved'}, 'to': 'save'},
                                  {'when': {'kind': 'rejected'}, 'to': 'cancelled'}]},
    'save': {'step': {'kind': 'tool_call', 'tool_id': 'create_record',
                        'args_template': {'collection': 'reports', 'data': '$report'},
                        'output_field': 'saved'},
              'transitions': [{'when': {'kind': 'tool_ok'}, 'to': 'done'},
                               {'when': {'kind': 'tool_failed'}, 'to': 'cancelled'}]},
    'done': {'step': {'kind': 'end', 'output_template': {'status': 'saved', 'report': '$report'}},
              'transitions': []},
    'cancelled': {'step': {'kind': 'end', 'output_template': {'status': 'cancelled'}},
                   'transitions': []}}})

lybo.register_skill(skill_id='field.report', name='Capture report',
                    workflow_id='wf.report', allowed_tools=['create_record'],
                    triggers=['file a report'], keywords=['report', 'file', 'log'])

out = lybo.submit(session, 'file a report asset_id: PUMP-7 condition: leaking seal')
print('state:', out['state'], '→', out.get('reason', ''))


In [ ]:
# The run is parked. YOU are the human in the loop — approve it.
# Note: there are TWO gates here by design — the workflow's confirm step,
# and a second one because create_record is a *sensitive* tool.
while out['state'] == 'awaiting_approval':
    print('approving:', out.get('reason'))
    out = lybo.resolve_approval(out['approval_id'], True)
print(out['state'], '→', out['output'])


## 4 · Train routing from labelled examples
`train_triggers` learns class-discriminative keywords (tf-idf across your labelled utterances). The phrases also build the **on-device semantic centroids**, so routing handles paraphrases the keywords miss.


In [ ]:
learned = train_triggers({
    'field.ask':    ['what does the manual say about pumps',
                      'search the safety guide for hot work',
                      'look up the torque specification'],
    'field.report': ['file a report asset_id PUMP-7',
                      'capture an inspection report',
                      'log a fault for tank two'],
})
for skill, cfg in learned.items():
    print(skill, '→ keywords:', cfg['keywords'])


## 5 · Build → sign → install → evaluate a capability pack
Packs are how agents ship and update **after app release**: ed25519-signed, declarative-only, verified on device. The eval suite is your CI gate.


In [ ]:
import json, subprocess, tempfile
keys = json.loads(subprocess.run([os.environ['LYBO_BIN'], 'keygen'],
                                  capture_output=True, text=True).stdout)

work = tempfile.mkdtemp(prefix='pack-')
src = (PackBuilder('colab.field', 'Field Assistant', '1.0.0', publisher='colab',
                    permissions=['records.write'])
    .add_knowledge('manual', 'Field Manual',
                   'The seal on pump unit seven must be replaced every six months using part S-42.')
    .add_skill('field.ask', 'Ask the manual', pattern='retrieval',
               triggers=learned['field.ask']['triggers'],
               keywords=learned['field.ask']['keywords'],
               knowledge_scopes=['colab.field'])
    .add_eval_case('suite', 'routes-and-answers',
                   'what does the manual say about the pump seal',
                   expect_skill='field.ask', expect_contains=['seal'])
    .save(f'{work}/pack.json'))
packed = sign_pack(src, keys['secret_key_hex'], f'{work}/field.lybopack',
                   lybo_binary=os.environ['LYBO_BIN'])

qa = Client(data_dir=tempfile.mkdtemp(prefix='qa-'), binary=os.environ['LYBO_BIN'],
            trust=[f"colab:{keys['public_key_hex']}"])
print('installed:', qa.install_pack_file(packed))
suite = json.load(open(src))['content']['eval_suites'][0]
report = evaluate(qa, suite)
print(report.summary())
assert report.ok


## 6 · Observability: privacy-safe export + tamper-evident audit trail
Traces never contain raw user content. Every stored trace hash-chains to the previous one — modifying or deleting any of them breaks verification.


In [ ]:
rows = export_routing_dataset(qa, f'{work}/routing.jsonl')
print(f'{rows} rows →', open(f'{work}/routing.jsonl').readline().strip())
print('audit chain intact:', qa.verify_trace_chain())
qa.close(); lybo.close()


## Next steps
- **Build visually, no code**: the hosted [Edge Studio Canvas](https://www.lyboai.app/studio) — design, eval, sign and publish packs in the browser.
- **Ship it in an app**: the same agent definitions run on-device via the Expo module — `npm i @lyboai/runtime expo-lyboai`.
- **Docs**: https://lyboai.app/developers/docs
